In [1]:
import numpy as np

x = np.arange(1_000_000)
%timeit x ** 2

5.8 ms ± 563 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [2]:

def square_loop(x):
    result = []
    for value in x:
        result.append(value ** 2)
    return result
    
%timeit square_loop(x)

445 ms ± 106 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [3]:
def square_numpy(x):
    return x ** 2

%timeit square_numpy(x)

7.74 ms ± 1.36 ms per loop (mean ± std. dev. of 7 runs, 100 loops each)


| Method | Time |
|---------|---------|
| Python loop | 210 ms ± 19.7 ms per loop |
| NumPy vectorization | 4.88 ms ± 325 μs per loop |

- Numpy vectorization was faster than the python loop.
- Speedup = loop time / Numpy Time
   ( 210 / 4.88 ) = 43.03
- numpy is faster because of compiled C loops, no per-element Python object overhead, contiguous typed memory 

In [4]:
x1 = np.arange(100)
x2 = np.arange(10_000)
x3 = np.arange(1_000_000)

%timeit x1 ** 2
%timeit x2 ** 2
%timeit x3 ** 2

1.95 μs ± 127 ns per loop (mean ± std. dev. of 7 runs, 1,000,000 loops each)
17.7 μs ± 227 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)
7.69 ms ± 1.21 ms per loop (mean ± std. dev. of 7 runs, 100 loops each)


**Explain:** time increases as the array gets larger, roughly in proportion to size (about 100 times more data, roughly 100 times more time, with small arrays dominated by fixed overhead).

In [5]:
data = np.random.normal(size=1_000_000)

%timeit np.mean(data)
%timeit np.std(data)
%timeit np.sum(data)

1.79 ms ± 50.7 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
11.1 ms ± 197 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
1.79 ms ± 76.7 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


- `np.sum` is faster
- **Why measure:** performance depends on hardware, software versions and data size, and intuition about what is slow is often wrong.


In [6]:
pip install line_profiler

Note: you may need to restart the kernel to use updated packages.


In [10]:
%load_ext line_profiler

from profiling_example import calculate_statistics

x = np.random.normal(size=1_000_000)

%lprun -f calculate_statistics calculate_statistics(x)

The line_profiler extension is already loaded. To reload it, use:
  %reload_ext line_profiler


Timer unit: 1e-07 s

Total time: 0.0176835 s
File: C:\Users\chimwemwe ponya\Desktop\icd341\labs\profiling_example.py
Function: calculate_statistics at line 3

Line #      Hits         Time  Per Hit   % Time  Line Contents
     3                                           def calculate_statistics(x):
     4         1      21661.0  21661.0     12.2      total = np.sum(x)
     5         1      22137.0  22137.0     12.5      mean = np.mean(x)
     6         1     132837.0 132837.0     75.1      variance = np.var(x)
     7         1        168.0    168.0      0.1      result = np.sqrt(variance)
     8                                           
     9         1         32.0     32.0      0.0      return total, mean, result

## Line Profiler Analysis

### 1. Which line takes the most time?

The line:

```python
variance = np.var(x)

## 2. How much time is spent on each line?

| Line | Code | Time | Percentage of Total Time |
|------|------|------:|------:|
| 4 | `total = np.sum(x)` | 21,661 | 12.2% |
| 5 | `mean = np.mean(x)` | 22,137 | 12.5% |
| 6 | `variance = np.var(x)` | 132,837 | 75.1% |
| 7 | `result = np.sqrt(variance)` | 168 | 0.1% |
| 9 | `return total, mean, result` | 32 | 0.0% |

The profiling results show that most of the execution time is spent calculating the variance, while the square root and return operations take very little time.

## 3. Which parts of the function might need optimization

In [12]:
total = np.sum(x)
mean = np.mean(x)

In [14]:

def slow_function(x):
    result = []
    for value in x:
        result.append(value ** 2)
    total = sum(result)
    return total

def fast_function(x):
    return np.sum(x ** 2)

x = np.arange(1_000_000)

In [15]:
%lprun -f slow_function slow_function(x)

Timer unit: 1e-07 s

Total time: 2.44898 s

Could not find file C:\Users\chimwemwe ponya\AppData\Local\Temp\ipykernel_13780\3744424222.py
Are you sure you are running this program from the same directory
that you ran the profiler from?
Continuing without the function's contents.

Line #      Hits         Time  Per Hit   % Time  Line Contents
     1                                           
     2         1         97.0     97.0      0.0  
     3   1000001   11063712.0     11.1     45.2  
     4   1000000   12488487.0     12.5     51.0  
     5         1     937476.0 937476.0      3.8  
     6         1         59.0     59.0      0.0

In [16]:
%timeit slow_function(x)
%timeit fast_function(x)

570 ms ± 75.2 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
9.66 ms ± 1.9 ms per loop (mean ± std. dev. of 7 runs, 100 loops each)


## Profiling Results

The `slow_function()` was profiled using `line_profiler`.

The most time-consuming lines were:

```python
result.append(value ** 2)

Profile `slow_function` with `%lprun`: the `for` loop and `append` lines dominate. Benchmark both with `%timeit`.

**Explain:** `fast_function` squares and sums in compiled code with no Python list of a million objects, so it is faster and uses less memory.


In [17]:
np.random.seed(42)
data = np.random.normal(
loc=100,
scale=20,
size=5_000_000
)

In [18]:
def squared_deviations_loop(x):
    mean = np.mean(x)
    total = 0

    for value in x:
        total += (value - mean) ** 2

    return total


def squared_deviations_numpy(x):
    mean = np.mean(x)
    return np.sum((x - mean) ** 2)

In [19]:
%timeit squared_deviations_loop(data)
%timeit squared_deviations_numpy(data)

5.31 s ± 836 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
93.1 ms ± 14.9 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


## Benchmark Results

| Method | Time |
|----------|----------|
| Loop Implementation | __________ |
| NumPy Implementation | __________ |

## Analysis

### 1. Which implementation was faster?

The NumPy implementation was faster than the loop implementation.

### 2. Why was it faster?

The NumPy implementation uses vectorized operations:

```python
np.sum((x - mean) ** 2)

3. Speed-up = loop time / NumPy time (e.g. 3 s / 0.03 s = 100x).
4. Vectorization removes interpreter overhead per element, uses optimised C, and works on contiguous memory (and can use CPU vector instructions).
5. Profiling shows which lines actually consume time, so you optimise the real bottleneck instead of guessing.